In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

RANDOM_STATE = 0

In [2]:
# Load dataset

In [3]:
df = pd.read_csv('../data/clean/cleaned_dataframe.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61069 entries, 0 to 61068
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            61069 non-null  int64  
 1   class                 61069 non-null  str    
 2   cap-color             61069 non-null  str    
 3   cap-shape             61069 non-null  str    
 4   does-bruise-or-bleed  61069 non-null  str    
 5   gill-color            61069 non-null  str    
 6   has-ring              61069 non-null  str    
 7   season                61069 non-null  str    
 8   habitat               61069 non-null  str    
 9   cap-diameter          61069 non-null  float64
 10  stem-height           61069 non-null  float64
 11  stem-width            61069 non-null  float64
dtypes: float64(3), int64(1), str(8)
memory usage: 5.6 MB


In [5]:
# Drop Unnamed column
df = df.drop(columns="Unnamed: 0")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61069 entries, 0 to 61068
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   class                 61069 non-null  str    
 1   cap-color             61069 non-null  str    
 2   cap-shape             61069 non-null  str    
 3   does-bruise-or-bleed  61069 non-null  str    
 4   gill-color            61069 non-null  str    
 5   has-ring              61069 non-null  str    
 6   season                61069 non-null  str    
 7   habitat               61069 non-null  str    
 8   cap-diameter          61069 non-null  float64
 9   stem-height           61069 non-null  float64
 10  stem-width            61069 non-null  float64
dtypes: float64(3), str(8)
memory usage: 5.1 MB


In [6]:
# We split the data into train and test

X_mush = df.drop(columns="class")
y_mush = df["class"]

X_train_mush, X_test_mush, y_train_mush, y_test_mush = train_test_split(
    X_mush,
    y_mush,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_mush,
)

print("Training observations:", len(X_train_mush))
print("Test observations:", len(X_test_mush))
print("Training features:", list(X_train_mush.columns))

Training observations: 48855
Test observations: 12214
Training features: ['cap-color', 'cap-shape', 'does-bruise-or-bleed', 'gill-color', 'has-ring', 'season', 'habitat', 'cap-diameter', 'stem-height', 'stem-width']


In [7]:
# Split features by type
numeric_features = ["cap-diameter", 'stem-height', 'stem-width']
nominal_features = [    "cap-color", "cap-shape","does-bruise-or-bleed", "gill-color", "has-ring", "season", "habitat", "cap-diameter", "stem-height", "stem-width"]


In [8]:
# Define the enconder for Nominal features

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train_mush[nominal_features])

encoded_train = pd.DataFrame(
    encoder.transform(X_train_mush[nominal_features]),
    columns=encoder.get_feature_names_out(),
    index=X_train_mush.index,
)
display(encoded_train.head())


,cap-color_b,cap-color_e,cap-color_g,cap-color_k,cap-color_l,cap-color_n,cap-color_o,cap-color_p,cap-color_r,cap-color_u,...,stem-width_88.16,stem-width_88.24,stem-width_89.09,stem-width_89.19,stem-width_89.45,stem-width_91.91,stem-width_92.52,stem-width_94.24,stem-width_95.68,stem-width_102.48
1117,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32619,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12883,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43050,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20204,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Creating a pipeline for Nominal features

nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

print(nominal_pipeline)

Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore'))])


In [10]:
# Scale Numerical features
scaler = StandardScaler()

scaled_train = pd.DataFrame(
    scaler.fit_transform(X_train_mush[numeric_features]),
    columns=numeric_features,
    index=X_train_mush.index,
)

scaled_test = pd.DataFrame(
    scaler.transform(X_test_mush[numeric_features]),
    columns=numeric_features,
    index=X_test_mush.index,
)

In [11]:
# Creating a pipeline for Numerical features

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', scaler),
])

print(numeric_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [12]:
# Combine nominal and numerical features

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("nominal", nominal_pipeline, nominal_features),
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

model.fit(X_train_mush, y_train_mush)
y_pred = model.predict(X_test_mush)

In [13]:
y_pred

array(['p', 'e', 'p', ..., 'e', 'e', 'p'], shape=(12214,), dtype=object)

In [14]:
# Evaluate predictions

print(classification_report(
    y_test_mush,
    y_pred,
    labels=["e", "p"],
    target_names=["Edible", "Poisonous"],
))

ConfusionMatrixDisplay.from_predictions(
    y_test_mush,
    y_pred,
    labels=["e", "p"],
    display_labels=["Edible", "Poisonous"],
    cmap="Blues",
)
plt.show()

              precision    recall  f1-score   support

      Edible       0.99      0.99      0.99      5436
   Poisonous       0.99      0.99      0.99      6778

    accuracy                           0.99     12214
   macro avg       0.99      0.99      0.99     12214
weighted avg       0.99      0.99      0.99     12214



NameError: name 'ConfusionMatrixDisplay' is not defined

In [15]:
# We want to compare the Test and Training models to check if there is overfitting or underfitting

train_accuracy = model.score(X_train_mush, y_train_mush)
test_accuracy = model.score(X_test_mush, y_test_mush)

print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Test accuracy:     {test_accuracy:.2%}")
print(f"Accuracy gap:     {(train_accuracy - test_accuracy) * 100:.2f} percentage points")

Training accuracy: 99.25%
Test accuracy:     98.98%
Accuracy gap:     0.26 percentage points
